# ByteNet 4b — Gradient-Free Training

This notebook trains the **ByteNet 4b** generation model using a **gradient-free Evolution Strategies (ES)** optimizer.

**Model: ByteNet 4b**  
- 4 TabNet decision steps (`n_steps=4`)  
- "b" (big) hidden dimensions: `n_d=512`, `n_a=512`, `hidden_dim=512`  
- 8 decoder layers, `max_seq_length=2048`  
- Byte-level tokenisation (vocab 320)

**Optimizer: OpenAI Evolution Strategies**  
The ES update never calls `jax.grad`. Instead it estimates the loss gradient via random parameter perturbations:

$$\hat{g} = \frac{1}{2 N \sigma} \sum_{i=1}^{N} \left[ F(\theta + \sigma \varepsilon_i) - F(\theta - \sigma \varepsilon_i) \right] \varepsilon_i$$

where $\varepsilon_i \sim \mathcal{N}(0, I)$ are antithetic noise samples and $F(\theta)$ is the batch loss.

The antithetic pairing halves the variance of the estimator at no extra cost.

## 1 — Setup

In [ ]:
# Install / upgrade required packages
# (skip this cell if your environment is already set up)
import subprocess, sys
pkgs = [
    "jax>=0.4.20",
    "jaxlib>=0.4.20",
    "equinox>=0.11.0",
    "optax>=0.1.7",
    "haliax>=1.3.0",
    "jaxtyping>=0.2.0",
    "datasets",
    "tqdm",
    "matplotlib",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx

# Make sure the ByteNet source is on the path
SRC_DIR = Path(".").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from byte_tabnet import (
    ByteTabNet,
    ByteTabNetSeq2Seq,
    ByteEmbedding,
    seq2seq_loss_with_tabnet_sparsity,
)
from tokenizer import ByteTokenizer

print(f"JAX backend : {jax.default_backend()}")
print(f"JAX devices : {jax.devices()}")
print(f"JAX version : {jax.__version__}")
print(f"Equinox     : {eqx.__version__}")

## 2 — ByteNet 4b Configuration

In [ ]:
@dataclass
class ByteNet4bConfig:
    """ByteNet 4b model + training hyper-parameters."""

    # ---- Model architecture ----
    vocab_size: int        = 320      # 256 bytes + special tokens
    max_seq_length: int    = 2048     # encoder context window
    embed_dim: int         = 64       # byte embedding dim
    hidden_dim: int        = 512      # TabNet hidden / decoder hidden
    n_steps: int           = 4        # 4 TabNet decision steps  ← "4"
    n_d: int               = 512      # decision embedding dim   ← "b" (big)
    n_a: int               = 512      # attention embedding dim  ← "b" (big)
    gamma: float           = 1.5      # TabNet relaxation factor
    n_shared: int          = 2        # shared GLU layers per FT block
    n_step_layers: int     = 4        # step-specific GLU layers per FT block
    virtual_batch_size: int= 32       # Ghost BN virtual batch size
    n_decoder_layers: int  = 8        # autoregressive decoder depth
    max_target_length: int = 1024     # decoder positional embedding length

    # ---- Dataset ----
    hf_dataset: str        = "toughdata/quora-question-answer-dataset"
    text_column: str       = "question"
    target_column: str     = "answer"
    max_examples: int      = 5_000    # cap for fast iteration in notebook
    max_src_len: int       = 256      # encoder tokens per example
    max_tgt_len: int       = 256      # decoder tokens per example

    # ---- ES Training ----
    seed: int              = 42
    n_es_samples: int      = 16       # antithetic pairs per ES step
    es_sigma: float        = 0.01     # ES perturbation std
    es_lr: float           = 0.01     # ES step size (Adam on ES gradient)
    batch_size: int        = 8        # examples per fitness evaluation
    num_steps: int         = 500      # total ES optimisation steps
    log_every: int         = 10       # print interval
    eval_every: int        = 50       # validation interval
    checkpoint_dir: str    = "./checkpoints/bytenet_4b_gf"
    save_every: int        = 100      # checkpoint every N steps


cfg = ByteNet4bConfig()

print("ByteNet 4b configuration")
print("=" * 50)
for k, v in asdict(cfg).items():
    print(f"  {k:30s}: {v}")

## 3 — Build the Model

In [ ]:
key = jrandom.PRNGKey(cfg.seed)
key, model_key = jrandom.split(key)

model = ByteTabNetSeq2Seq(
    vocab_size        = cfg.vocab_size,
    max_seq_length    = cfg.max_seq_length,
    embed_dim         = cfg.embed_dim,
    hidden_dim        = cfg.hidden_dim,
    n_steps           = cfg.n_steps,
    n_d               = cfg.n_d,
    n_a               = cfg.n_a,
    gamma             = cfg.gamma,
    n_decoder_layers  = cfg.n_decoder_layers,
    virtual_batch_size= cfg.virtual_batch_size,
    max_target_length = cfg.max_target_length,
    key               = model_key,
)

# ---- Count parameters ----
leaves = jax.tree_util.tree_leaves(eqx.filter(model, eqx.is_array))
total_params = sum(l.size for l in leaves)
print(f"\nByteNet 4b — total parameters: {total_params:,}")
print(f"  (~{total_params / 1e6:.1f}M parameters)")
print()

# Quick sanity-check forward pass
dummy_src = jnp.ones((2, 32), dtype=jnp.int32)
dummy_tgt = jnp.ones((2, 16), dtype=jnp.int32)
logits, masks, _ = model(dummy_src, dummy_tgt, inference=True)
print(f"Logits shape : {logits.shape}  — expected (2, 16, {cfg.vocab_size})")
print(f"Masks  shape : {masks.shape}")

## 4 — Dataset

In [ ]:
try:
    from datasets import load_dataset
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    warnings.warn("HuggingFace `datasets` not available — using synthetic data.")


def _make_synthetic_pairs(n: int = 500) -> Tuple[List[str], List[str]]:
    """Tiny synthetic QA pairs for offline use."""
    nouns = ["Python", "JAX", "machine learning", "TabNet", "bytes",
             "attention", "gradient", "neural network", "optimiser", "loss"]
    sources, targets = [], []
    rng = np.random.default_rng(0)
    for i in range(n):
        noun = nouns[i % len(nouns)]
        sources.append(f"What is {noun} and how does it work?")
        targets.append(
            f"{noun} is a concept in computing and statistics. "
            f"It involves transformations and iterative refinement. "
            f"Example {i}: the key property of {noun} is expressiveness."
        )
    return sources, targets


def load_qa_pairs(
    cfg: ByteNet4bConfig,
) -> Tuple[List[str], List[str]]:
    """Load source/target pairs from HuggingFace or synthetic fallback."""
    if HF_AVAILABLE:
        print(f"Loading dataset: {cfg.hf_dataset} …")
        try:
            ds = load_dataset(cfg.hf_dataset, split="train")
            sources = [str(r) for r in ds[cfg.text_column]  if r]
            targets = [str(r) for r in ds[cfg.target_column] if r]
            # Zip to keep pairs aligned
            pairs = [(s, t) for s, t in zip(sources, targets)
                     if s.strip() and t.strip()]
            pairs = pairs[:cfg.max_examples]
            sources, targets = zip(*pairs) if pairs else ([], [])
            sources, targets = list(sources), list(targets)
            print(f"Loaded {len(sources)} QA pairs from HuggingFace.")
            return sources, targets
        except Exception as exc:
            print(f"HuggingFace load failed ({exc}). Using synthetic data.")

    print("Using synthetic QA pairs.")
    return _make_synthetic_pairs(cfg.max_examples)


sources_all, targets_all = load_qa_pairs(cfg)
N = len(sources_all)

# Train / val split (90 / 10)
split = int(N * 0.9)
train_src, val_src = sources_all[:split], sources_all[split:]
train_tgt, val_tgt = targets_all[:split], targets_all[split:]

print(f"\nTrain : {len(train_src)} pairs")
print(f"Val   : {len(val_src)} pairs")
print(f"\nSample source : {train_src[0][:120]}")
print(f"Sample target : {train_tgt[0][:120]}")

In [ ]:
tokenizer = ByteTokenizer()


def encode_batch(
    src_texts: List[str],
    tgt_texts: List[str],
    max_src: int,
    max_tgt: int,
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Encode a list of source/target pairs into padded JAX arrays."""
    src_ids, src_mask = tokenizer.encode_batch(
        src_texts, max_length=max_src, add_bos=True, add_eos=False
    )
    tgt_ids, tgt_mask = tokenizer.encode_batch(
        tgt_texts, max_length=max_tgt, add_bos=True, add_eos=True
    )
    return src_ids, src_mask, tgt_ids, tgt_mask


def make_batches(
    src: List[str],
    tgt: List[str],
    batch_size: int,
    max_src: int,
    max_tgt: int,
    shuffle: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """Yield encoded batches."""
    indices = np.arange(len(src))
    if shuffle:
        (rng or np.random.default_rng()).shuffle(indices)
    for start in range(0, len(src) - batch_size + 1, batch_size):
        idx = indices[start : start + batch_size]
        yield encode_batch(
            [src[i] for i in idx],
            [tgt[i] for i in idx],
            max_src, max_tgt,
        )


# Verify encoding
sample_batch = next(make_batches(
    train_src, train_tgt,
    batch_size=4,
    max_src=cfg.max_src_len,
    max_tgt=cfg.max_tgt_len,
    shuffle=False,
))
s_ids, s_mask, t_ids, t_mask = sample_batch
print(f"Source ids  shape : {s_ids.shape}")
print(f"Source mask shape : {s_mask.shape}")
print(f"Target ids  shape : {t_ids.shape}")
print(f"Target mask shape : {t_mask.shape}")

## 5 — Gradient-Free Evolution Strategies Optimizer

### Why gradient-free?

Gradient-free methods are useful when:
- The loss is **non-differentiable** (e.g., BLEU, exact-match accuracy).
- Backpropagation is unavailable (e.g., black-box model deployment).
- You want to explore a **diversity of solutions** that gradient descent misses.
- The parameter space has **many local minima** and ES's inherent noise helps escape them.

### Algorithm: Antithetic OpenAI ES

```
for step = 1, 2, ..., T:
    Sample ε₁, …, ε_N  ~  N(0, I)         # N perturbations
    For each i:
        F⁺ᵢ = loss(θ + σ·εᵢ, batch)       # positive perturbation
        F⁻ᵢ = loss(θ - σ·εᵢ, batch)       # antithetic (negative)
    ĝ = (1 / 2Nσ) · Σᵢ (F⁻ᵢ - F⁺ᵢ)·εᵢ  # gradient estimate
    θ ← θ - α·ĝ                            # Adam or SGD step
```

The antithetic sign flip `(F⁻ - F⁺)` is used because we are **minimising** the loss: a perturbation that **decreases** the loss (`F⁺ < F⁻`) should push `θ` in the direction of `+ε`.

In [ ]:
# ============================================================================
# Flatten / unflatten helpers
# ============================================================================

# Partition model into trainable arrays vs. static structure
_params, _static = eqx.partition(model, eqx.is_array)

# Flatten all leaf arrays into a single 1-D vector
_flat_init, _unflatten_fn = jax.flatten_util.ravel_pytree(_params)

print(f"Flat parameter vector length: {len(_flat_init):,}")


def params_to_model(flat: jnp.ndarray, static) -> ByteTabNetSeq2Seq:
    """Reconstruct a full model from a flat parameter vector."""
    params = _unflatten_fn(flat)
    return eqx.combine(params, static)


def model_to_flat(mdl: ByteTabNetSeq2Seq) -> jnp.ndarray:
    """Extract a flat parameter vector from a model."""
    p, _ = eqx.partition(mdl, eqx.is_array)
    flat, _ = jax.flatten_util.ravel_pytree(p)
    return flat

In [ ]:
# ============================================================================
# Batch loss function (gradient-free — no jax.grad)
# ============================================================================

@eqx.filter_jit
def batch_loss(
    flat_params: jnp.ndarray,
    static,
    src_ids:  jnp.ndarray,
    src_mask: jnp.ndarray,
    tgt_ids:  jnp.ndarray,
    tgt_mask: jnp.ndarray,
) -> jnp.ndarray:
    """Evaluate the seq2seq loss for a flat parameter vector — no gradients."""
    mdl = params_to_model(flat_params, static)

    logits, encoder_masks, _ = mdl(
        src_ids, tgt_ids,
        attention_mask=src_mask,
        decoder_attention_mask=tgt_mask,
        inference=False,
    )

    loss = seq2seq_loss_with_tabnet_sparsity(
        logits[:, :-1],
        tgt_ids[:, 1:],
        encoder_masks,
        tgt_mask[:, 1:],
        sparsity_weight=1e-3,
        label_smoothing=0.1,
    )
    return loss


# Quick check (should return a scalar)
s_ids, s_mask, t_ids, t_mask = sample_batch
init_loss = batch_loss(_flat_init, _static, s_ids, s_mask, t_ids, t_mask)
print(f"Initial batch loss : {float(init_loss):.4f}")

In [ ]:
# ============================================================================
# Antithetic Evolution Strategies — ES gradient estimator
# ============================================================================

@eqx.filter_jit
def es_gradient_estimate(
    flat_params: jnp.ndarray,
    static,
    batch: Tuple,
    key: jnp.ndarray,
    n_samples: int,
    sigma: float,
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """
    Estimate the ES gradient using antithetic sampling.

    Returns
    -------
    grad_estimate : flat gradient vector (same shape as flat_params)
    mean_loss     : mean of (F⁺ + F⁻) / 2  across all pairs
    fitness_std   : std of antithetic fitness differences (diagnostic)
    """
    src_ids, src_mask, tgt_ids, tgt_mask = batch
    n = flat_params.shape[0]

    # Sample N perturbations  ε_i ~ N(0, I)
    keys = jrandom.split(key, n_samples)
    noises = jax.vmap(lambda k: jrandom.normal(k, (n,)))(keys)  # (N, n)

    # Evaluate loss at θ + σε  and  θ - σε
    def eval_pair(noise):
        f_pos = batch_loss(
            flat_params + sigma * noise, static,
            src_ids, src_mask, tgt_ids, tgt_mask,
        )
        f_neg = batch_loss(
            flat_params - sigma * noise, static,
            src_ids, src_mask, tgt_ids, tgt_mask,
        )
        return f_pos, f_neg

    f_pos_all, f_neg_all = jax.vmap(eval_pair)(noises)  # each (N,)

    # Antithetic fitness difference: (F⁻ - F⁺) pushes toward ε when F⁺ < F⁻
    antithetic_diff = f_neg_all - f_pos_all  # (N,)  — positive = good perturbation

    # Rank-normalise to reduce sensitivity to loss scale
    ranks = jnp.argsort(jnp.argsort(antithetic_diff))
    normalised = (ranks.astype(jnp.float32) / (n_samples - 1)) - 0.5  # in [-0.5, 0.5]

    # Gradient estimate: ĝ = (1 / 2Nσ) Σ (F⁻ - F⁺)·ε
    # With rank normalisation: ĝ = (1 / N·σ) Σ rank_norm·ε
    grad_estimate = jnp.mean(
        normalised[:, None] * noises, axis=0
    ) / sigma  # (n,)

    mean_loss = jnp.mean((f_pos_all + f_neg_all) / 2.0)
    fitness_std = jnp.std(antithetic_diff)

    return grad_estimate, mean_loss, fitness_std


print("es_gradient_estimate compiled.")

In [ ]:
# ============================================================================
# Adam-on-ES-gradient state
# ============================================================================

import optax

class ESOptimizerState:
    """Wraps flat parameters and an optax Adam state for ES updates."""

    def __init__(self, flat_params: jnp.ndarray, lr: float = 0.01):
        self.flat_params = flat_params
        self.optimizer   = optax.adam(learning_rate=lr)
        self.opt_state   = self.optimizer.init(flat_params)
        self.step        = 0

    def apply_gradient(self, grad: jnp.ndarray):
        """Apply one Adam step with the ES gradient estimate."""
        updates, self.opt_state = self.optimizer.update(grad, self.opt_state)
        self.flat_params = optax.apply_updates(self.flat_params, updates)
        self.step += 1


es_opt = ESOptimizerState(flat_params=_flat_init, lr=cfg.es_lr)
print(f"ESOptimizerState ready. Params: {es_opt.flat_params.shape}")

## 6 — Checkpoint Utilities

In [ ]:
from pathlib import Path

ckpt_dir = Path(cfg.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)


def save_checkpoint(
    flat_params: jnp.ndarray,
    step: int,
    loss: float,
    tag: str = "",
):
    """Serialise the current flat parameter vector."""
    mdl = params_to_model(flat_params, _static)
    suffix = f"_step_{step}" if not tag else f"_{tag}"
    path = ckpt_dir / f"bytenet_4b{suffix}.eqx"
    eqx.tree_serialise_leaves(path, mdl)
    meta = {"step": step, "loss": loss, "config": asdict(cfg)}
    with open(path.with_suffix(".json"), "w") as f:
        json.dump(meta, f, indent=2)
    return path


def load_checkpoint(path: str) -> jnp.ndarray:
    """Reload a checkpoint and return its flat parameter vector."""
    mdl_loaded = eqx.tree_deserialise_leaves(path, model)
    return model_to_flat(mdl_loaded)


print(f"Checkpoints will be saved to: {ckpt_dir.resolve()}")

## 7 — Training Loop

In [ ]:
def compute_val_loss(
    flat_params: jnp.ndarray,
    val_src: List[str],
    val_tgt: List[str],
    max_batches: int = 10,
) -> float:
    """Average loss on up to `max_batches` validation batches."""
    total, count = 0.0, 0
    for i, batch in enumerate(
        make_batches(val_src, val_tgt, cfg.batch_size,
                     cfg.max_src_len, cfg.max_tgt_len, shuffle=False)
    ):
        if i >= max_batches:
            break
        s_ids, s_mask, t_ids, t_mask = batch
        total += float(batch_loss(flat_params, _static, s_ids, s_mask, t_ids, t_mask))
        count += 1
    return total / max(count, 1)


print(f"Initial validation loss: {compute_val_loss(es_opt.flat_params, val_src, val_tgt):.4f}")

In [ ]:
# ============================================================================
# Main ES training loop
# ============================================================================

history = {
    "step":      [],
    "train_loss": [],
    "val_loss":   [],
    "fitness_std": [],
    "elapsed_s":  [],
}

best_val_loss  = float("inf")
best_flat      = es_opt.flat_params

data_rng   = np.random.default_rng(cfg.seed + 1)
key        = jrandom.PRNGKey(cfg.seed + 2)
t0         = time.time()

# Infinite cycling batch iterator
_batch_iter = make_batches(
    train_src, train_tgt,
    cfg.batch_size, cfg.max_src_len, cfg.max_tgt_len,
    shuffle=True, rng=data_rng,
)

def _next_batch():
    global _batch_iter, data_rng
    try:
        return next(_batch_iter)
    except StopIteration:
        # Reshuffle and restart
        _batch_iter = make_batches(
            train_src, train_tgt,
            cfg.batch_size, cfg.max_src_len, cfg.max_tgt_len,
            shuffle=True, rng=data_rng,
        )
        return next(_batch_iter)


pbar = tqdm(range(1, cfg.num_steps + 1), desc="ES training")

for step in pbar:
    batch = _next_batch()
    key, es_key = jrandom.split(key)

    grad, mean_loss, fitness_std = es_gradient_estimate(
        es_opt.flat_params,
        _static,
        batch,
        es_key,
        cfg.n_es_samples,
        cfg.es_sigma,
    )

    es_opt.apply_gradient(grad)

    # ---- Logging ----
    if step % cfg.log_every == 0:
        elapsed = time.time() - t0
        history["step"].append(step)
        history["train_loss"].append(float(mean_loss))
        history["fitness_std"].append(float(fitness_std))
        history["elapsed_s"].append(elapsed)

        pbar.set_postfix({
            "loss": f"{float(mean_loss):.4f}",
            "σ(F)": f"{float(fitness_std):.4f}",
        })

    # ---- Validation ----
    if step % cfg.eval_every == 0:
        v_loss = compute_val_loss(es_opt.flat_params, val_src, val_tgt)
        history["val_loss"].append((step, v_loss))
        print(f"  [step {step:4d}]  train={float(mean_loss):.4f}  val={v_loss:.4f}")

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_flat = es_opt.flat_params
            save_checkpoint(best_flat, step, v_loss, tag="best")
            print(f"  ★ New best model — val loss {v_loss:.4f}")

    # ---- Periodic checkpoint ----
    if step % cfg.save_every == 0:
        save_checkpoint(es_opt.flat_params, step, float(mean_loss))


total_time = time.time() - t0
print(f"\nTraining complete in {total_time/60:.1f} min")
print(f"Best validation loss : {best_val_loss:.4f}")

## 8 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 8.1 Train loss
ax = axes[0]
ax.plot(history["step"], history["train_loss"], color="steelblue", linewidth=1.5)
ax.set_title("ES Mean Batch Loss (train)")
ax.set_xlabel("ES step")
ax.set_ylabel("loss")
ax.grid(True, alpha=0.3)

# 8.2 Validation loss
ax = axes[1]
if history["val_loss"]:
    vsteps, vlosses = zip(*history["val_loss"])
    ax.plot(vsteps, vlosses, color="darkorange", marker="o", linewidth=1.5)
    ax.axhline(best_val_loss, color="red", linestyle="--", label=f"best={best_val_loss:.4f}")
    ax.legend()
ax.set_title("Validation Loss")
ax.set_xlabel("ES step")
ax.set_ylabel("loss")
ax.grid(True, alpha=0.3)

# 8.3 Fitness std (exploration)
ax = axes[2]
ax.plot(history["step"], history["fitness_std"], color="seagreen", linewidth=1.5)
ax.set_title("Fitness Std  (ES exploration signal)")
ax.set_xlabel("ES step")
ax.set_ylabel("std(F⁻ − F⁺)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ckpt_dir / "training_curves.png", dpi=150)
plt.show()
print("Saved training_curves.png")

## 9 — Text Generation with the Best Model

In [ ]:
def generate_answer(
    question: str,
    flat_params: jnp.ndarray,
    max_length: int = 128,
    temperature: float = 0.8,
    top_k: int = 50,
    seed: int = 0,
) -> str:
    """Generate an answer to `question` using the best-checkpoint model."""
    mdl = params_to_model(flat_params, _static)

    src_ids, src_mask = tokenizer.encode_batch(
        [question],
        max_length=cfg.max_src_len,
        add_bos=True,
        add_eos=False,
    )

    gen_ids = mdl.generate(
        src_ids,
        attention_mask=src_mask,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        do_sample=True,
        key=jrandom.PRNGKey(seed),
    )

    return tokenizer.decode(gen_ids[0])


# Test on a few held-out questions
test_questions = [
    val_src[0] if val_src else "What is machine learning?",
    val_src[1] if len(val_src) > 1 else "How does attention work?",
    "What are the advantages of gradient-free optimisation?",
]

print("=" * 70)
print("Generation examples (best checkpoint)")
print("=" * 70)
for i, q in enumerate(test_questions):
    answer = generate_answer(q, best_flat, seed=i)
    ref = val_tgt[i] if i < len(val_tgt) else "(no reference)"
    print(f"\n[{i+1}] QUESTION  : {q[:100]}")
    print(f"    REFERENCE : {ref[:120]}")
    print(f"    GENERATED : {answer[:120]}")
print("=" * 70)

## 10 — Evaluation Metrics

In [ ]:
def compute_perplexity(
    flat_params: jnp.ndarray,
    src_list: List[str],
    tgt_list: List[str],
    max_batches: int = 20,
) -> float:
    """Token-level perplexity on a held-out set."""
    total_ce, total_tokens = 0.0, 0
    mdl = params_to_model(flat_params, _static)

    for i, batch in enumerate(
        make_batches(src_list, tgt_list, cfg.batch_size,
                     cfg.max_src_len, cfg.max_tgt_len, shuffle=False)
    ):
        if i >= max_batches:
            break
        s_ids, s_mask, t_ids, t_mask = batch

        logits, _, _ = mdl(
            s_ids, t_ids,
            attention_mask=s_mask,
            decoder_attention_mask=t_mask,
            inference=True,
        )

        shifted_logits  = logits[:, :-1]
        shifted_targets = t_ids[:, 1:]
        shifted_mask    = t_mask[:, 1:]

        log_probs = jax.nn.log_softmax(shifted_logits, axis=-1)
        token_lp  = jnp.take_along_axis(
            log_probs, shifted_targets[..., None], axis=-1
        ).squeeze(-1)

        total_ce     += float((-token_lp * shifted_mask).sum())
        total_tokens += int(shifted_mask.sum())

    avg_ce     = total_ce / max(total_tokens, 1)
    perplexity = float(np.exp(min(avg_ce, 100)))
    return perplexity


# Compare initial model vs best checkpoint
init_ppl  = compute_perplexity(_flat_init,  val_src, val_tgt, max_batches=5)
best_ppl  = compute_perplexity(best_flat,   val_src, val_tgt, max_batches=10)

print(f"Perplexity — initial model : {init_ppl:.2f}")
print(f"Perplexity — best model    : {best_ppl:.2f}")

if best_ppl < init_ppl:
    rel_imp = 100 * (init_ppl - best_ppl) / init_ppl
    print(f"  ↓ {rel_imp:.1f}% relative improvement")
else:
    print("  Model did not improve over random init (try more ES steps).")

## 11 — Hyperparameter Analysis: σ Sensitivity

In [ ]:
# Quick sweep over ES sigma to show its effect on gradient quality
sigmas       = [0.001, 0.005, 0.01, 0.05, 0.1]
sigma_losses = []

probe_batch = next(make_batches(
    val_src, val_tgt, 4,
    cfg.max_src_len, cfg.max_tgt_len, shuffle=False,
))
probe_key = jrandom.PRNGKey(999)

for sigma in sigmas:
    _, m_loss, f_std = es_gradient_estimate(
        best_flat, _static, probe_batch, probe_key,
        n_samples=8, sigma=sigma,
    )
    sigma_losses.append((sigma, float(m_loss), float(f_std)))
    print(f"  σ={sigma:.3f}  mean_loss={float(m_loss):.4f}  fitness_std={float(f_std):.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(
    [r[0] for r in sigma_losses],
    [r[2] for r in sigma_losses],
    "o-", color="purple", linewidth=2,
)
ax.set_title("ES Fitness Std vs Perturbation σ")
ax.set_xlabel("σ (perturbation std)")
ax.set_ylabel("std(F⁻ − F⁺)")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 12 — Resume / Load Best Checkpoint

In [ ]:
best_ckpt_path = ckpt_dir / "bytenet_4b_best.eqx"

if best_ckpt_path.exists():
    loaded_flat = load_checkpoint(str(best_ckpt_path))
    loaded_ppl  = compute_perplexity(loaded_flat, val_src, val_tgt, max_batches=5)
    print(f"Loaded checkpoint: {best_ckpt_path}")
    print(f"Loaded model perplexity: {loaded_ppl:.2f}")

    # Optionally continue training from the loaded checkpoint
    # es_opt = ESOptimizerState(flat_params=loaded_flat, lr=cfg.es_lr)
    # ... run more training steps ...
else:
    print("No best checkpoint found (run training cells first).")

## 13 — Export Results Summary

In [ ]:
summary = {
    "model": "ByteNet 4b",
    "optimizer": "Antithetic OpenAI ES + Adam",
    "config": asdict(cfg),
    "total_params": total_params,
    "training_steps": cfg.num_steps,
    "best_val_loss": best_val_loss,
    "initial_perplexity": init_ppl,
    "best_perplexity": best_ppl,
    "training_time_s": total_time,
    "history": history,
    "val_loss_trace": history["val_loss"],
}

summary_path = ckpt_dir / "training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Summary written to: {summary_path}")
print()
print("=" * 55)
print(" ByteNet 4b — Gradient-Free Training Summary")
print("=" * 55)
print(f"  Model parameters    : {total_params:,}")
print(f"  ES steps completed  : {cfg.num_steps}")
print(f"  ES samples / step   : {cfg.n_es_samples} pairs (antithetic)")
print(f"  ES sigma            : {cfg.es_sigma}")
print(f"  ES learning rate    : {cfg.es_lr}")
print(f"  Best val loss       : {best_val_loss:.4f}")
print(f"  Initial perplexity  : {init_ppl:.2f}")
print(f"  Best perplexity     : {best_ppl:.2f}")
print(f"  Training time       : {total_time/60:.1f} min")
print("=" * 55)

---

## Notes

### Gradient-Free vs Gradient-Based

| Aspect | Gradient-Based (AdamW) | Gradient-Free (ES) |
|---|---|---|
| Gradient computation | `jax.grad` / backprop | Random perturbations |
| Memory | O(params) extra for grads | O(N·params) for N samples |
| Signal-to-noise | High (exact gradient) | Low (estimated) |
| Parallelism | 1 forward+backward | N independent forwards |
| Non-diff. losses | ✗ | ✓ |
| Black-box models | ✗ | ✓ |

### Tuning Tips

- **`n_es_samples`**: Higher → better gradient estimate, but N× more compute. Use 16–64.
- **`es_sigma`**: Too small → no signal; too large → fitness collapses. Start at `0.01`.
- **`es_lr`**: The Adam LR on the ES gradient is independent of `sigma`. Start at `0.01`.
- **Batch size**: Larger batches reduce noise in the fitness evaluation, producing cleaner ES gradients.
- **Rank normalisation**: Used above to decouple the gradient estimate from the absolute scale of the loss — helps stability across different `sigma` values.

### Scaling ES

For very large models, ES can be embarrassingly parallelised:  
all `2N` fitness evaluations are independent and can be spread across devices with `jax.pmap` or distributed with Ray/Jax multi-host.